# NB09b — Model Diagnostic & Comparison Charts

Generates Plotly charts for the blog statistical analysis section:
1. OLS coefficient forest plot
2. Residual diagnostics (residuals vs fitted, Q-Q, histogram)
3. Model comparison (base vs enriched, + Ridge/Lasso)
4. Logistic regression ROC curve

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parents[1]
CHART_DIR = PROJECT_ROOT / 'data' / 'outputs' / 'blog_charts'
CHART_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Rebuild models from data (same pipeline as NB10)
# ============================================================
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import roc_curve, roc_auc_score, mean_squared_error, r2_score

# Load enriched data
enriched_path = PROJECT_ROOT / 'data' / 'outputs' / 'nb10a_additional_features' / 'hospital_enriched_features.csv'
if enriched_path.exists():
    df = pd.read_csv(enriched_path, dtype={'ccn': str})
else:
    df = pd.read_csv(PROJECT_ROOT / 'data' / 'outputs' / 'nb06_peer_benchmarks' / 'hospital_with_benchmarks.csv', dtype={'ccn': str})

gap_scores = pd.read_csv(PROJECT_ROOT / 'data' / 'outputs' / 'nb07_gap_scores' / 'hospital_gap_scores.csv', dtype={'ccn': str})
df['ccn'] = df['ccn'].str.zfill(6)
gap_scores['ccn'] = gap_scores['ccn'].str.zfill(6)
score_cols = [c for c in gap_scores.columns if c not in df.columns]
score_cols.append('ccn')
df = df.merge(gap_scores[score_cols], on='ccn', how='left')

model_df = df.dropna(subset=['doc_gap_score', 'cmi', 'ownership_category']).copy()

# Prepare regression data
base_required = ['doc_gap_score', 'beds', 'wage_index', 'dsh_pct',
                 'resident_to_bed_ratio', 'ownership_category', 'is_teaching', 'is_urban']
reg_df = model_df.dropna(subset=base_required).copy()
reg_df['log_beds'] = np.log1p(reg_df['beds'])
reg_df['is_nonprofit'] = (reg_df['ownership_category'] == 'Nonprofit').astype(int)
reg_df['is_government'] = (reg_df['ownership_category'] == 'Government').astype(int)
reg_df['is_other'] = (reg_df['ownership_category'] == 'Other').astype(int)
reg_df['teaching'] = reg_df['is_teaching'].astype(int)
reg_df['urban'] = reg_df['is_urban'].astype(int)
reg_df['drg_diversity_filled'] = reg_df['drg_diversity'].fillna(reg_df['drg_diversity'].median()) if 'drg_diversity' in reg_df.columns else 0

# Base features
base_cols = ['is_nonprofit', 'is_government', 'is_other', 'teaching',
             'resident_to_bed_ratio', 'log_beds', 'urban', 'wage_index',
             'dsh_pct', 'drg_diversity_filled']
base_labels = ['Nonprofit', 'Government', 'Other Ownership',
               'Teaching Hospital', 'Resident/Bed Ratio', 'Log(Beds)',
               'Urban', 'Wage Index', 'DSH %', 'DRG Diversity']

# Enriched features
enriched_cols = list(base_cols)
enriched_labels = list(base_labels)
for col, label in [('clinical_staff_per_bed', 'Clinical Staff/Bed'),
                    ('rn_skill_mix', 'RN Skill Mix'),
                    ('physician_per_bed', 'Physician/Bed'),
                    ('star_rating', 'Star Rating'),
                    ('readm_pct_worse', 'Readmit % Worse'),
                    ('safety_pct_worse', 'Safety % Worse'),
                    ('quality_score', 'Quality Score')]:
    if col in reg_df.columns and reg_df[col].notna().sum() >= 100:
        fill_col = f'{col}_filled'
        reg_df[fill_col] = reg_df[col].fillna(reg_df[col].median())
        enriched_cols.append(fill_col)
        enriched_labels.append(label)

y = reg_df['doc_gap_score'].values

# Fit enriched OLS
scaler = StandardScaler()
X_enriched = scaler.fit_transform(reg_df[enriched_cols].values)
ols = LinearRegression().fit(X_enriched, y)
y_pred = ols.predict(X_enriched)
residuals = y - y_pred

# Compute p-values
n, p = X_enriched.shape
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y - y.mean())**2)
mse = ss_res / (n - p - 1)
X_int = np.column_stack([np.ones(n), X_enriched])
var_beta = mse * np.linalg.inv(X_int.T @ X_int).diagonal()
se_beta = np.sqrt(var_beta)
coefs_all = np.concatenate([[ols.intercept_], ols.coef_])
t_stats = coefs_all / se_beta
p_values = 2 * stats.t.sf(np.abs(t_stats), df=n - p - 1)

print(f'Enriched OLS: R²={1-ss_res/ss_tot:.4f}, n={n}, features={p}')
print(f'Residuals: mean={residuals.mean():.4f}, std={residuals.std():.2f}')

Enriched OLS: R²=0.2064, n=3060, features=17
Residuals: mean=-0.0000, std=10.30


## Chart 8: OLS Coefficient Forest Plot

In [2]:
# Forest plot of standardized coefficients with 95% CI
coef_vals = ols.coef_
coef_se = se_beta[1:]  # skip intercept
ci_lower = coef_vals - 1.96 * coef_se
ci_upper = coef_vals + 1.96 * coef_se
pvals = p_values[1:]

# Sort by absolute coefficient
sort_idx = np.argsort(np.abs(coef_vals))
sorted_labels = [enriched_labels[i] for i in sort_idx]
sorted_coefs = coef_vals[sort_idx]
sorted_ci_low = ci_lower[sort_idx]
sorted_ci_up = ci_upper[sort_idx]
sorted_pvals = pvals[sort_idx]

# Color by significance
colors = ['#2E86AB' if pv < 0.05 else '#cccccc' for pv in sorted_pvals]

fig8 = go.Figure()

# Error bars (CI)
fig8.add_trace(go.Scatter(
    x=sorted_coefs, y=sorted_labels,
    mode='markers',
    marker=dict(size=10, color=colors, line=dict(width=1, color='#333')),
    error_x=dict(
        type='data',
        symmetric=False,
        array=sorted_ci_up - sorted_coefs,
        arrayminus=sorted_coefs - sorted_ci_low,
        color='#888', thickness=1.5, width=5
    ),
    hovertemplate='%{y}<br>Coef: %{x:.3f}<br>CI: [%{customdata[0]:.3f}, %{customdata[1]:.3f}]<br>p=%{customdata[2]:.4f}<extra></extra>',
    customdata=np.column_stack([sorted_ci_low, sorted_ci_up, sorted_pvals])
))

# Zero line
fig8.add_vline(x=0, line_dash='dash', line_color='#999', line_width=1)

fig8.update_layout(
    title=dict(text='OLS Regression: Standardized Coefficients with 95% CI', font=dict(size=16)),
    xaxis_title='Standardized Coefficient (1 SD change in gap score)',
    height=500, width=850,
    margin=dict(l=160, r=40, t=60, b=60),
    plot_bgcolor='white',
    showlegend=False,
    xaxis=dict(gridcolor='#eee', zeroline=True, zerolinecolor='#999'),
    yaxis=dict(gridcolor='#f5f5f5'),
    annotations=[dict(x=0.99, y=-0.12, xref='paper', yref='paper',
                      text='Blue = significant (p<0.05) | Gray = not significant',
                      showarrow=False, font=dict(size=11, color='#888'))]
)

fig8.write_html(CHART_DIR / 'chart8_ols_coefficients.html', include_plotlyjs='cdn', full_html=False)
fig8.show()
print('Saved chart8_ols_coefficients.html')

Saved chart8_ols_coefficients.html


## Chart 9: Residual Diagnostics (2x2 panel)

In [3]:
# 2x2 diagnostic panel
fig9 = make_subplots(rows=2, cols=2,
    subplot_titles=('Residuals vs Fitted', 'Q-Q Plot',
                    'Residual Histogram', 'Scale-Location'),
    horizontal_spacing=0.12, vertical_spacing=0.14)

# 1. Residuals vs Fitted
fig9.add_trace(go.Scatter(
    x=y_pred, y=residuals, mode='markers',
    marker=dict(size=3, color='#2E86AB', opacity=0.4),
    showlegend=False
), row=1, col=1)
fig9.add_hline(y=0, line_dash='dash', line_color='red', line_width=1, row=1, col=1)

# 2. Q-Q Plot
standardized_res = (residuals - residuals.mean()) / residuals.std()
theoretical_q = np.sort(stats.norm.ppf(np.linspace(0.001, 0.999, len(standardized_res))))
empirical_q = np.sort(standardized_res)
# Subsample for performance
step = max(1, len(theoretical_q) // 500)
fig9.add_trace(go.Scatter(
    x=theoretical_q[::step], y=empirical_q[::step], mode='markers',
    marker=dict(size=3, color='#2E86AB', opacity=0.5),
    showlegend=False
), row=1, col=2)
qq_min, qq_max = min(theoretical_q), max(theoretical_q)
fig9.add_trace(go.Scatter(
    x=[qq_min, qq_max], y=[qq_min, qq_max],
    mode='lines', line=dict(color='red', dash='dash', width=1),
    showlegend=False
), row=1, col=2)

# 3. Residual Histogram
fig9.add_trace(go.Histogram(
    x=residuals, nbinsx=50, marker_color='#2E86AB', opacity=0.7,
    showlegend=False
), row=2, col=1)

# 4. Scale-Location (sqrt of abs standardized residuals vs fitted)
sqrt_abs_std_res = np.sqrt(np.abs(standardized_res))
fig9.add_trace(go.Scatter(
    x=y_pred, y=sqrt_abs_std_res, mode='markers',
    marker=dict(size=3, color='#2E86AB', opacity=0.4),
    showlegend=False
), row=2, col=2)

# LOWESS for residuals vs fitted
from statsmodels.nonparametric.smoothers_lowess import lowess as sm_lowess
try:
    lowess_fit = sm_lowess(residuals, y_pred, frac=0.3)
    fig9.add_trace(go.Scatter(
        x=lowess_fit[:, 0], y=lowess_fit[:, 1],
        mode='lines', line=dict(color='red', width=2),
        showlegend=False
    ), row=1, col=1)
except:
    pass

fig9.update_xaxes(title_text='Fitted Values', row=1, col=1)
fig9.update_yaxes(title_text='Residuals', row=1, col=1)
fig9.update_xaxes(title_text='Theoretical Quantiles', row=1, col=2)
fig9.update_yaxes(title_text='Sample Quantiles', row=1, col=2)
fig9.update_xaxes(title_text='Residual Value', row=2, col=1)
fig9.update_yaxes(title_text='Count', row=2, col=1)
fig9.update_xaxes(title_text='Fitted Values', row=2, col=2)
fig9.update_yaxes(title_text='√|Standardized Residuals|', row=2, col=2)

fig9.update_layout(
    title=dict(text='OLS Regression Diagnostics', font=dict(size=16)),
    height=600, width=900,
    plot_bgcolor='white',
    margin=dict(t=80, b=60)
)

fig9.write_html(CHART_DIR / 'chart9_ols_diagnostics.html', include_plotlyjs='cdn', full_html=False)
fig9.show()
print('Saved chart9_ols_diagnostics.html')

Saved chart9_ols_diagnostics.html


## Chart 10: Model Comparison (5-fold CV)

In [4]:
# Compare OLS base, OLS enriched, Ridge, Lasso, ElasticNet using 5-fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Base model
X_base = StandardScaler().fit_transform(reg_df[base_cols].values)

models = {
    'OLS (Base, 10 features)': (LinearRegression(), X_base),
    'OLS (Enriched, 17 features)': (LinearRegression(), X_enriched),
    'Ridge (Enriched, α=1.0)': (Ridge(alpha=1.0), X_enriched),
    'Ridge (Enriched, α=10.0)': (Ridge(alpha=10.0), X_enriched),
    'Lasso (Enriched, α=0.1)': (Lasso(alpha=0.1, max_iter=5000), X_enriched),
    'Lasso (Enriched, α=1.0)': (Lasso(alpha=1.0, max_iter=5000), X_enriched),
    'ElasticNet (Enriched)': (ElasticNet(alpha=0.5, l1_ratio=0.5, max_iter=5000), X_enriched),
}

results = {}
for name, (model, X) in models.items():
    cv_r2 = cross_val_score(model, X, y, cv=kf, scoring='r2')
    cv_rmse = -cross_val_score(model, X, y, cv=kf, scoring='neg_root_mean_squared_error')
    # Train R²
    model.fit(X, y)
    train_r2 = r2_score(y, model.predict(X))
    results[name] = {
        'cv_r2_mean': cv_r2.mean(), 'cv_r2_std': cv_r2.std(),
        'cv_rmse_mean': cv_rmse.mean(), 'cv_rmse_std': cv_rmse.std(),
        'train_r2': train_r2,
        'n_coefs': np.sum(np.abs(model.coef_) > 1e-6) if hasattr(model, 'coef_') else X.shape[1]
    }
    print(f'{name:40s}  CV-R²={cv_r2.mean():.4f}±{cv_r2.std():.4f}  CV-RMSE={cv_rmse.mean():.2f}±{cv_rmse.std():.2f}  Train-R²={train_r2:.4f}  Active={results[name]["n_coefs"]}')

# Build comparison chart
model_names = list(results.keys())
cv_r2_means = [results[m]['cv_r2_mean'] for m in model_names]
cv_r2_stds = [results[m]['cv_r2_std'] for m in model_names]
train_r2s = [results[m]['train_r2'] for m in model_names]

fig10 = go.Figure()

fig10.add_trace(go.Bar(
    x=model_names, y=train_r2s,
    name='Train R²',
    marker_color='#2E86AB', opacity=0.5,
    text=[f'{v:.4f}' for v in train_r2s],
    textposition='outside', textfont=dict(size=10)
))

fig10.add_trace(go.Bar(
    x=model_names, y=cv_r2_means,
    name='5-Fold CV R²',
    marker_color='#E8553D',
    error_y=dict(type='data', array=cv_r2_stds, visible=True, color='#333'),
    text=[f'{v:.4f}' for v in cv_r2_means],
    textposition='outside', textfont=dict(size=10)
))

fig10.update_layout(
    title=dict(text='Model Comparison: Train vs 5-Fold Cross-Validated R²', font=dict(size=16)),
    yaxis_title='R²',
    barmode='group',
    height=500, width=900,
    plot_bgcolor='white',
    margin=dict(b=140, t=60),
    xaxis=dict(tickangle=-30),
    yaxis=dict(gridcolor='#eee', range=[0, max(train_r2s)*1.3]),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)')
)

fig10.write_html(CHART_DIR / 'chart10_model_comparison.html', include_plotlyjs='cdn', full_html=False)
fig10.show()
print('Saved chart10_model_comparison.html')

OLS (Base, 10 features)                   CV-R²=0.1712±0.0290  CV-RMSE=10.51±0.09  Train-R²=0.1807  Active=10
OLS (Enriched, 17 features)               CV-R²=0.1896±0.0214  CV-RMSE=10.40±0.06  Train-R²=0.2064  Active=17
Ridge (Enriched, α=1.0)                   CV-R²=0.1896±0.0213  CV-RMSE=10.40±0.06  Train-R²=0.2064  Active=17
Ridge (Enriched, α=10.0)                  CV-R²=0.1898±0.0212  CV-RMSE=10.40±0.06  Train-R²=0.2063  Active=17
Lasso (Enriched, α=0.1)                   CV-R²=0.1919±0.0203  CV-RMSE=10.38±0.08  Train-R²=0.2042  Active=14
Lasso (Enriched, α=1.0)                   CV-R²=0.1362±0.0160  CV-RMSE=10.74±0.13  Train-R²=0.1396  Active=2


ElasticNet (Enriched)                     CV-R²=0.1672±0.0168  CV-RMSE=10.54±0.09  Train-R²=0.1750  Active=10


Saved chart10_model_comparison.html


## Chart 11: ROC Curve (Logistic Regression)

In [5]:
# Logistic regression ROC
reg_df['is_high_gap'] = (reg_df['doc_gap_score'] >= 65).astype(int)
y_binary = reg_df['is_high_gap'].values

logit = LogisticRegression(max_iter=1000, random_state=42)
logit.fit(X_enriched, y_binary)
y_prob = logit.predict_proba(X_enriched)[:, 1]
auc = roc_auc_score(y_binary, y_prob)
fpr, tpr, thresholds = roc_curve(y_binary, y_prob)

fig11 = go.Figure()

fig11.add_trace(go.Scatter(
    x=fpr, y=tpr, mode='lines',
    line=dict(color='#2E86AB', width=2.5),
    name=f'Logistic Regression (AUC = {auc:.3f})',
    fill='tozeroy', fillcolor='rgba(46,134,171,0.1)'
))

fig11.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines',
    line=dict(color='#999', dash='dash', width=1),
    name='Random Classifier'
))

fig11.update_layout(
    title=dict(text=f'ROC Curve: Predicting High-Gap Hospitals (AUC = {auc:.3f})', font=dict(size=16)),
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    height=480, width=550,
    plot_bgcolor='white',
    xaxis=dict(gridcolor='#eee', constrain='domain'),
    yaxis=dict(gridcolor='#eee', scaleanchor='x'),
    legend=dict(x=0.35, y=0.05),
    margin=dict(l=60, r=40, t=60, b=60)
)

fig11.write_html(CHART_DIR / 'chart11_roc_curve.html', include_plotlyjs='cdn', full_html=False)
fig11.show()
print(f'Saved chart11_roc_curve.html  |  AUC={auc:.4f}')

Saved chart11_roc_curve.html  |  AUC=0.7533


## Summary

In [6]:
import os
print('Blog chart files:')
for f in sorted(os.listdir(CHART_DIR)):
    size = os.path.getsize(CHART_DIR / f) / 1024
    print(f'  {f:45s} {size:7.1f} KB')

# Best model summary
best_model = max(results.items(), key=lambda x: x[1]['cv_r2_mean'])
print(f'\nBest model by CV-R²: {best_model[0]}')
print(f'  CV R² = {best_model[1]["cv_r2_mean"]:.4f} ± {best_model[1]["cv_r2_std"]:.4f}')
print(f'  Train R² = {best_model[1]["train_r2"]:.4f}')
print(f'  Active coefficients: {best_model[1]["n_coefs"]}')

Blog chart files:
  chart10_model_comparison.html                     9.0 KB
  chart11_roc_curve.html                           19.3 KB
  chart1_cmi_by_ownership.html                     87.4 KB
  chart2_severity_national.html                     7.9 KB
  chart3_gap_score_distribution.html               44.4 KB
  chart4_revenue_by_size.html                       8.2 KB
  chart5_cmi_vs_gap.html                          251.7 KB
  chart6_revenue_by_state.html                      8.3 KB
  chart7_severity_uplift.html                       8.8 KB
  chart8_ols_coefficients.html                     10.3 KB
  chart9_ols_diagnostics.html                     275.1 KB

Best model by CV-R²: Lasso (Enriched, α=0.1)
  CV R² = 0.1919 ± 0.0203
  Train R² = 0.2042
  Active coefficients: 14
